# Cafeteria Order Data – Quick Analysis & 7-Day Forecast Challenge
**Organization:** Kanishka Software Pvt Ltd  
**Dataset Scope:** Enterprise POS & Mobile Cafeteria System across 18 Branches (FY 2024–2025)  
**Deliverable:** Exploratory Data Analysis (EDA), Peak Hour Profiling, Menu Performance & 7-Day Order Demand Forecasting

---
### Notebook Contents
1. **Environment Setup & Data Loading**
2. **Data Integrity & Reconciliation**
3. **Exploratory Data Analysis (EDA)**
   - Branch Performance & Market Concentration
   - Ordering Channel Dynamics (Mobile vs SOK vs POS)
   - Payment Mode Adoption
   - Operating Rush Hours & Meal Schedules
   - Menu Engineering & Counter Revenue
   - Customer Frequency & Retention Cohorts
4. **Time Series Modeling & 7-Day Forecast (Selected Branch: Embassy Tech Village)**
   - Baseline Model (Seasonal Naive)
   - Holt-Winters Exponential Smoothing
   - SARIMAX(1,1,1)(1,1,1,7)
   - Machine Learning (Random Forest & Gradient Boosting Regressors)
   - Test Holdout Performance Comparison
   - Out-of-Sample 7-Day Forecast with 95% Confidence Intervals
5. **Business & Operational Recommendations**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 120

print("All required analytical libraries imported successfully.")


## 1. Load Processed Enterprise Data
The raw source is an ~11 GB mysqldump containing over 5.96 million orders and 7.42 million line items across the entire financial year. To enable fast, reproducible in-memory analysis, data has been parsed into clean structured datasets.


In [ ]:
# Load pre-processed datasets
daily_df = pd.read_csv('../data/processed/daily_branch_orders.csv')
hourly_df = pd.read_csv('../data/processed/hourly_sales_pattern.csv')
pay_df = pd.read_csv('../data/processed/payment_methods.csv')
chan_df = pd.read_csv('../data/processed/order_channels.csv')
dishes_df = pd.read_csv('../data/processed/top_dishes.csv')
cntr_df = pd.read_csv('../data/processed/counter_sales.csv')
cust_df = pd.read_csv('../data/processed/customer_frequency.csv')
branches_df = pd.read_csv('../data/processed/branches.csv')

daily_df['date'] = pd.to_datetime(daily_df['date'])

print(f"Daily Branch Orders Shape: {daily_df.shape}")
print(f"Dishes Catalog Shape:       {dishes_df.shape}")
print(f"Customer Profiles:          {cust_df.shape[0]:,} transacting users")
daily_df.head(5)


## 2. Portfolio Executive Summary & Financial Reconciliation
We reconcile total transaction counts, revenue, taxes, and average order value (AOV) across the entire FY 2024-2025 dataset.


In [ ]:
total_orders = daily_df['total_orders'].sum()
total_revenue = daily_df['revenue'].sum()
total_tax = daily_df['tax'].sum()
overall_aov = total_revenue / total_orders

summary_table = pd.DataFrame({
    'Metric': [
        'Date Span',
        'Total Tracked Days',
        'Active Branches Analyzed',
        'Total Completed Orders',
        'Gross Revenue (INR)',
        'Total Tax Collected (INR)',
        'Average Order Value (AOV)'
    ],
    'Value': [
        f"{daily_df['date'].min().strftime('%Y-%m-%d')} to {daily_df['date'].max().strftime('%Y-%m-%d')}",
        f"{(daily_df['date'].max() - daily_df['date'].min()).days + 1} days",
        f"{daily_df['branch_id'].nunique()} branches",
        f"{total_orders:,}",
        f"₹{total_revenue:,.2f} (~₹{total_revenue/1e7:.2f} Cr)",
        f"₹{total_tax:,.2f}",
        f"₹{overall_aov:.2f}"
    ]
})
summary_table


## 3. Branch Performance & Market Concentration
Analysis of order volume and revenue generation across all corporate tech park branches.


In [ ]:
branch_summary = daily_df.groupby(['branch_id', 'branch_name']).agg(
    total_orders=('total_orders', 'sum'),
    total_revenue=('revenue', 'sum'),
    avg_daily_orders=('total_orders', 'mean'),
    avg_aov=('avg_order_value', 'mean')
).reset_index().sort_values('total_orders', ascending=False)

branch_summary['order_share_%'] = (branch_summary['total_orders'] / total_orders) * 100
branch_summary['revenue_share_%'] = (branch_summary['total_revenue'] / total_revenue) * 100

# Format display
styled_branches = branch_summary.copy()
styled_branches['total_orders'] = styled_branches['total_orders'].map('{:,}'.format)
styled_branches['total_revenue'] = styled_branches['total_revenue'].map('₹{:,.2f}'.format)
styled_branches['avg_daily_orders'] = styled_branches['avg_daily_orders'].round(1)
styled_branches['avg_aov'] = styled_branches['avg_aov'].map('₹{:.2f}'.format)
styled_branches['order_share_%'] = styled_branches['order_share_%'].map('{:.1f}%'.format)
styled_branches['revenue_share_%'] = styled_branches['revenue_share_%'].map('{:.1f}%'.format)

styled_branches


## 4. Ordering Channels & Payment Modernization
Evaluating user adoption across Self-Ordering Kiosks (SOK), Mobile App, and traditional POS counters, alongside digital payment method penetration.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

# Channel Share
ch_agg = chan_df.groupby('order_through')['order_count'].sum().sort_values(ascending=False)
ch_labels = ['Mobile App' if 'mobile' in k else 'SOK Kiosk' if 'sok' in k else 'POS Counter' for k in ch_agg.index]
colors_ch = ['#1976d2', '#ff9800', '#4caf50', '#9c27b0']

ax1.pie(ch_agg.values, labels=ch_labels, autopct='%1.1f%%', colors=colors_ch[:len(ch_agg)],
        startangle=140, pctdistance=0.75, wedgeprops=dict(width=0.45, edgecolor='w'))
ax1.set_title('Ordering Channel Split\n(Mobile App vs Kiosk vs POS)', fontsize=13, fontweight='bold')

# Payment Modes
pay_agg = pay_df.groupby('mode_of_transaction')['order_count'].sum().sort_values(ascending=False).head(5)
colors_pay = ['#00897b', '#f4511e', '#3949ab', '#7cb342', '#8e24aa']

ax2.pie(pay_agg.values, labels=pay_agg.index, autopct='%1.1f%%', colors=colors_pay[:len(pay_agg)],
        startangle=140, pctdistance=0.75, wedgeprops=dict(width=0.45, edgecolor='w'))
ax2.set_title('Payment Method Adoption\n(Top 5 Transaction Modes)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()


## 5. Hourly Cafeteria Traffic Profiling
Identifying operational rush hours to optimize kitchen staffing, batch food prep, and counter queue management.


In [ ]:
hour_summary = hourly_df.groupby('hour')['order_count'].sum().reset_index()
hour_summary['orders_thousands'] = hour_summary['order_count'] / 1000

plt.figure(figsize=(13, 5.5))
bars = plt.bar(hour_summary['hour'], hour_summary['orders_thousands'], color='#0288d1', alpha=0.7, width=0.7)
plt.plot(hour_summary['hour'], hour_summary['orders_thousands'], color='#d81b60', marker='o', linewidth=2)

# Shading rush windows
plt.axvspan(8.5, 10.5, color='#ffe082', alpha=0.35, label='Breakfast Peak (9:00 - 10:30 AM)')
plt.axvspan(12.0, 14.5, color='#ffab91', alpha=0.35, label='Lunch Peak (12:00 - 2:30 PM)')
plt.axvspan(16.5, 18.5, color='#c5cae9', alpha=0.35, label='Evening Snacks / Chai (4:30 - 6:30 PM)')

plt.xticks(range(24), [f'{h:02d}:00' for h in range(24)], rotation=45)
plt.title('Hourly Cafeteria Traffic Distribution (Meal Service Windows)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Hour of Day', fontweight='bold')
plt.ylabel('Annual Orders (in Thousands)', fontweight='bold')
plt.legend(frameon=True, facecolor='white', framealpha=0.95)
plt.tight_layout()
plt.show()


## 6. Menu Engineering: Top 15 Best-Selling Items
Examining high-velocity volume drivers (Chai, Coffee, Dosa) vs high-margin revenue drivers (Thalis, Combos).


In [ ]:
top15 = dishes_df.head(15).copy()
top15['revenue_lakhs'] = top15['total_revenue'] / 100000

plt.figure(figsize=(12, 7))
y_pos = np.arange(len(top15))
plt.barh(y_pos, top15['total_quantity_sold'] / 1000, color='#388e3c', alpha=0.85, height=0.65)
plt.yticks(y_pos, top15['dish_name'], fontsize=10)
plt.gca().invert_yaxis()
plt.xlabel('Total Units Sold (Thousands)', fontweight='bold')
plt.title('Top 15 Most Ordered Cafeteria Menu Items', fontsize=14, fontweight='bold', pad=15)

for i, (units, rev, prc) in enumerate(zip(top15['total_quantity_sold'], top15['revenue_lakhs'], top15['avg_dish_price'])):
    plt.text(units/1000 + 5, i, f"{units/1000:.1f}k units | ₹{rev:.1f}L (@ ₹{prc:.0f})", 
             va='center', fontsize=9, fontweight='semibold', color='#1b5e20')

plt.xlim(0, max(top15['total_quantity_sold']/1000) * 1.35)
plt.tight_layout()
plt.show()


## 7. Predictive Modeling: 7-Day Order Demand Forecast
We select **Embassy Tech Village (Branch 2)**, the largest cafeteria location with 2.48M annual orders, to build and benchmark predictive time-series models.

### Validation Methodology
- **Training Period:** 2024-04-01 to 2025-03-18 (352 days)
- **Holdout Test Set:** 2025-03-19 to 2025-04-01 (14 days)
- **Evaluation Metrics:** MAE, RMSE, MAPE (%)


In [ ]:
# Load model evaluation metrics and predictions
metrics_df = pd.read_csv('../data/processed/model_evaluation_metrics.csv')
test_eval_df = pd.read_csv('../data/processed/test_predictions.csv')
forecast_df = pd.read_csv('../data/processed/forecast_results.csv')

metrics_df.sort_values('MAE')


### Visualization of Model Performance & 7-Day Forward Forecast
Comparing test backtest actuals vs predictions, alongside the out-of-sample forward forecast with 95% confidence intervals.


In [ ]:
test_eval_df['date'] = pd.to_datetime(test_eval_df['date'])
forecast_df['date'] = pd.to_datetime(forecast_df['date'])

plt.figure(figsize=(13, 6))
plt.plot(test_eval_df['date'], test_eval_df['actual_orders'], color='black', marker='o', linewidth=2.5, label='Actual Orders (Holdout Test)')
plt.plot(test_eval_df['date'], test_eval_df['SARIMAX(1,1,1)(1,1,1,7)'], color='#e53935', linestyle='--', marker='x', label='SARIMAX Backtest')
plt.plot(test_eval_df['date'], test_eval_df['Random Forest Regressor'], color='#3949ab', linestyle=':', marker='^', label='Random Forest Backtest')

# 7-day Forward Forecast
plt.plot(forecast_df['date'], forecast_df['forecast_orders'], color='#2e7d32', marker='D', linewidth=3, label='7-Day Forward Forecast')
plt.fill_between(forecast_df['date'], forecast_df['lower_ci_95'], forecast_df['upper_ci_95'], color='#81c784', alpha=0.35, label='95% Prediction Interval')

plt.axvline(test_eval_df['date'].max(), color='#546e7a', linestyle='-', linewidth=1.5)
plt.title('7-Day Out-of-Sample Demand Forecast: Embassy Tech Village', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Date', fontweight='bold')
plt.ylabel('Daily Orders', fontweight='bold')
plt.xticks(rotation=45)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()


### Operational 7-Day Order Demand Schedule


In [ ]:
styled_forecast = forecast_df.copy()
styled_forecast['forecast_orders'] = styled_forecast['forecast_orders'].map('{:,}'.format)
styled_forecast['lower_ci_95'] = styled_forecast['lower_ci_95'].map('{:,}'.format)
styled_forecast['upper_ci_95'] = styled_forecast['upper_ci_95'].map('{:,}'.format)
styled_forecast['estimated_revenue_inr'] = styled_forecast['estimated_revenue_inr'].map('₹{:,.2f}'.format)
styled_forecast


## 8. Strategic Business & Operational Recommendations
Based on the empirical findings from 5.96 million orders across FY 2024–2025:

1. **Hybrid Workplace Staffing Alignment:**
   - **Tuesdays and Wednesdays** consistently peak at 8,300–9,500 daily orders, whereas **Fridays** taper by ~15–20% and **weekends** drop by 92%.
   - Staffing and raw ingredient procurement should follow a **3-tier dynamic schedule**: Tier A (Tue-Thu full staff), Tier B (Mon/Fri standard staff), Tier C (Sat/Sun skeleton crew).
2. **Express Queues for Beverage Counters:**
   - Beverage items (Ginger Tea, Cutting Chai, Filter Kaapi) account for over **1.6 million units** sold.
   - Operating dedicated "Beverage Express" pick-up lanes during the 4:30–6:30 PM evening surge will alleviate cafeteria bottlenecks.
3. **Menu Margin Optimization:**
   - Thalis and Veg Meal Combos account for the bulk of high-ticket revenue (₹7M–₹8M each).
   - Bundle high-margin beverages (₹15 Chai) with lunch combos at an introductory discount (e.g. ₹5 off) to boost average order value from ₹68.96 to ₹80.00.
4. **Mobile Ordering Incentives for Rush Reduction:**
   - Mobile app adoption is already strong at 60.2%. Incentivizing pre-ordering during 12:00–12:30 PM (prior to the 1:00 PM peak) with loyalty points will spread peak kitchen load evenly.
